# Notebook 7 — Full RMP: F duplication + light pruning + LoRA recovery

Notebook 6 established: **F (duplicate layers 9-34, 26-layer span) beats dense baseline by +0.20pp** on MMLU. That was on an intact model. Notebook 5 established that **LoRA recovery on Taylor-50% pruned was net −0.69pp** because the pruned base was at random chance.

This notebook combines both axes at *light* pruning levels where the model still has headroom:

**Phase A** — F + varying Taylor pruning (no LoRA). Measures raw accuracy vs sparsity with the duplication gain from F. Pruning levels: {10%, 20%, 30%, 50%}.

**Phase B** — F + best Taylor X% + LoRA recovery. Takes whichever Taylor X% + F config from Phase A looks most promising and adds a short LoRA fine-tune (r=16, AdamW full-precision, 500 MMLU-aux × 3 epochs — the only config notebook 5 showed was safe).

The goal: is there a light-pruning sweet spot where F duplication keeps MMLU near dense, and LoRA on top recovers the last pruning damage? That's the full RMP thesis — prune dead weight for compute savings, duplicate the circuit to keep or improve accuracy, fine-tune to clean up.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, time
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

import config
from eval_mmlu import evaluate, print_results

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
TILE_R, TILE_C = config.TILE_SIZES[0]
F_RANGE = (9, 34)  # The winning duplication range from notebook 6 — stitched_top_n(9)
                   # spans layers 9-34 (26-layer dup, 62 total). Earlier session logs
                   # had mislabeled this as (8, 16); see FINDINGS section 17.
PRUNE_LEVELS = [0.10, 0.20, 0.30, 0.50]

torch.set_float32_matmul_precision("high")
print(f"torch: {torch.__version__}  device: {torch.cuda.get_device_name(0)}")
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.2f} / {total/1e9:.2f} GB")

torch: 2.6.0+cu124  device: NVIDIA GeForce RTX 4070 Laptop GPU
GPU free: 7.40 / 8.18 GB


## 1. Load model, cache ORIGINAL_LAYERS + ORIGINAL_MLP_WEIGHTS

We'll apply and un-apply pruning multiple times (one per Taylor level). Cache the clean MLP weights on CPU so we can restore between tests without reloading the whole model.

In [2]:
print(f"loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=getattr(torch, config.TORCH_DTYPE),
    device_map=config.DEVICE,
)
model.eval()

ORIGINAL_LAYERS = list(model.model.layers)
N_LAYERS = len(ORIGINAL_LAYERS)
ORIGINAL_LAYER_TYPES = (list(model.config.layer_types)
                       if getattr(model.config, "layer_types", None) is not None else None)

# Cache clean MLP weights on CPU — used to 'un-prune' between Taylor X% tests
def is_mlp_name(name):
    return any(k in name for k in config.PRUNE_TARGETS_PATTERNS)

ORIGINAL_MLP_WEIGHTS = {}
for name, module in model.named_modules():
    if isinstance(module, nn.Linear) and is_mlp_name(name):
        ORIGINAL_MLP_WEIGHTS[name + ".weight"] = module.weight.data.detach().clone().cpu()
cpu_bytes = sum(t.numel() * t.element_size() for t in ORIGINAL_MLP_WEIGHTS.values())
print(f"cached {len(ORIGINAL_MLP_WEIGHTS)} MLP matrices on CPU ({cpu_bytes/1e9:.2f} GB)")
print(f"params: {sum(p.numel() for p in model.parameters())/1e9:.2f}B, layers: {N_LAYERS}")
print(f"peak GPU: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

loading Qwen/Qwen2.5-3B-Instruct...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

cached 108 MLP matrices on CPU (4.87 GB)
params: 3.09B, layers: 36
peak GPU: 6.22 GB


## 2. Calibration data (C4, 512 samples)

In [3]:
N_CAL = 512
SEQ_LEN_CAL = 128

cal_ds = load_dataset(config.CALIBRATION_DATASET, config.CALIBRATION_SUBSET,
                     split="train", streaming=True, trust_remote_code=True)
cal_texts = []
for ex in cal_ds:
    t = ex.get("text", "")
    if len(t) > 50:
        cal_texts.append(t)
    if len(cal_texts) >= N_CAL:
        break
cal_encodings = [tokenizer(t, return_tensors="pt", max_length=SEQ_LEN_CAL, truncation=True)
                 for t in cal_texts]
print(f"calibration samples: {len(cal_encodings)}")

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

calibration samples: 512


## 3. Taylor importance scoring

Same as notebooks 5/6.

In [4]:
def get_component_type(name):
    for p in config.PRUNE_TARGETS_PATTERNS:
        if p in name:
            return p
    return "unknown"

def get_layer_idx(name):
    parts = name.split(".")
    i = parts.index("layers")
    return int(parts[i+1])

for p in model.parameters():
    p.requires_grad_(False)
for name, p in model.named_parameters():
    if is_mlp_name(name) and p.ndim == 2:
        p.requires_grad_(True)

model.gradient_checkpointing_enable()
model.config.use_cache = False

importance_gpu = {}
def make_hook(pname):
    def hook(param):
        if param.grad is None:
            return
        taylor = (param.data * param.grad).abs().float()
        out_dim, in_dim = taylor.shape
        nr, nc = out_dim // TILE_R, in_dim // TILE_C
        tt = taylor[:nr*TILE_R, :nc*TILE_C].reshape(nr, TILE_R, nc, TILE_C).permute(0, 2, 1, 3)
        tile_scores = tt.reshape(nr, nc, -1).sum(dim=-1)
        if pname in importance_gpu:
            importance_gpu[pname] += tile_scores
        else:
            importance_gpu[pname] = tile_scores.clone()
        param.grad = None
    return hook

hook_handles = []
for name, p in model.named_parameters():
    if is_mlp_name(name) and p.ndim == 2 and p.requires_grad:
        hook_handles.append(p.register_post_accumulate_grad_hook(make_hook(name)))

print("Taylor calibration (fwd+bwd)...")
n_samples = 0
for enc in tqdm(cal_encodings, desc="taylor"):
    model.zero_grad(set_to_none=True)
    inputs = {k: v.to(config.DEVICE) for k, v in enc.items()}
    out = model(**inputs, labels=inputs["input_ids"])
    out.loss.backward()
    n_samples += 1

for h in hook_handles:
    h.remove()
model.gradient_checkpointing_disable()
model.config.use_cache = True
for p in model.parameters():
    p.requires_grad_(False)

importance_taylor = {name: (v / n_samples).cpu().numpy() for name, v in importance_gpu.items()}
del importance_gpu
gc.collect(); torch.cuda.empty_cache()

# Per-component-type z-normalization
comp_stats = {}
for ct in config.PRUNE_TARGETS_PATTERNS:
    vals = np.concatenate([m.ravel() for n, m in importance_taylor.items() if ct in n])
    comp_stats[ct] = (vals.mean(), vals.std())

importance_norm = {}
for name, m in importance_taylor.items():
    mu, sd = comp_stats[get_component_type(name)]
    importance_norm[name] = (m - mu) / sd if sd > 1e-8 else np.zeros_like(m)

# Per-layer aggregation — sanity check the F range is still [8,16]
layer_scores = np.zeros(N_LAYERS)
for name, m in importance_norm.items():
    layer_scores[get_layer_idx(name)] += m.sum()

top_layer = int(np.argmax(layer_scores))
print(f"peak layer: {top_layer} (expected near 11)")
print(f"layers 8-16 scores: {layer_scores[8:17].round(0).astype(int).tolist()}")
print(f"peak GPU during calibration: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

Taylor calibration (fwd+bwd)...


taylor: 100%|██████████████████████████████████████████████████████| 512/512 [03:26<00:00,  2.48it/s]


peak layer: 11 (expected near 11)
layers 8-16 scores: [2255, 3389, 4186, 4822, 3473, 4269, 3713, 3239, 2318]
peak GPU during calibration: 7.01 GB


## 4. Utilities — pruning, duplication, weight restore

`apply_taylor_masks(sparsity)` → zero the bottom-sparsity fraction of each MLP matrix in place (Taylor-ranked, per-matrix uniform, matching notebook 5/6).

`restore_mlp_weights()` → copy cached CPU weights back to GPU, un-pruning the model.

`set_F()` / `reset_layers()` → duplicate [8,16] as a contiguous block (the winning F range from notebook 6) / restore 36-layer stack.

In [5]:
# ── pruning / restore ─────────────────────────────────────────────────

def build_masks(sparsity):
    masks = {}
    for name, m in importance_norm.items():
        thr = float(np.percentile(m.ravel(), sparsity * 100))
        masks[name] = m < thr
    return masks

def apply_taylor_masks(sparsity):
    masks = build_masks(sparsity)
    with torch.no_grad():
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                pn = name + ".weight"
                if pn in masks:
                    mask = masks[pn]
                    w = module.weight.data
                    out_f, in_f = w.shape
                    nr, nc = out_f // TILE_R, in_f // TILE_C
                    for i in range(nr):
                        for j in range(nc):
                            if mask[i, j]:
                                w[i*TILE_R:(i+1)*TILE_R, j*TILE_C:(j+1)*TILE_C] = 0
    return masks

def restore_mlp_weights():
    with torch.no_grad():
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                pn = name + ".weight"
                if pn in ORIGINAL_MLP_WEIGHTS:
                    module.weight.data.copy_(ORIGINAL_MLP_WEIGHTS[pn].to(module.weight.device))

# ── duplication ───────────────────────────────────────────────────────

def set_duplication_range(start, end):
    new_layers = (list(ORIGINAL_LAYERS[:end+1])
                  + list(ORIGINAL_LAYERS[start:end+1])
                  + list(ORIGINAL_LAYERS[end+1:]))
    model.model.layers = nn.ModuleList(new_layers)
    model.config.num_hidden_layers = len(new_layers)
    if ORIGINAL_LAYER_TYPES is not None:
        model.config.layer_types = (ORIGINAL_LAYER_TYPES[:end+1]
                                    + ORIGINAL_LAYER_TYPES[start:end+1]
                                    + ORIGINAL_LAYER_TYPES[end+1:])

def reset_layers():
    model.model.layers = nn.ModuleList(ORIGINAL_LAYERS)
    model.config.num_hidden_layers = len(ORIGINAL_LAYERS)
    if ORIGINAL_LAYER_TYPES is not None:
        model.config.layer_types = list(ORIGINAL_LAYER_TYPES)

def set_F():
    set_duplication_range(*F_RANGE)

# ── eval wrapper ──────────────────────────────────────────────────────

def eval_tag(tag):
    t0 = time.time()
    results = evaluate(model, tokenizer, tag=tag)
    acc = results["overall"]["accuracy"]
    print(f"  {tag:<34s} MMLU {acc*100:5.2f}%   ({time.time()-t0:.1f}s)")
    return acc, results

# Smoke test
set_F()
print(f"set_F: {len(model.model.layers)} layers (expect 45)")
reset_layers()
print(f"reset: {len(model.model.layers)} layers (expect 36)")
restore_mlp_weights()
sample = next(iter(ORIGINAL_MLP_WEIGHTS.keys()))
sample_mod = dict(model.named_modules())[sample[:-len(".weight")]]
print(f"restore smoke: sample zero fraction = {100*(sample_mod.weight.data == 0).float().mean().item():.2f}%")

set_F: 45 layers (expect 45)
reset: 36 layers (expect 36)
restore smoke: sample zero fraction = 0.00%


## 5. Phase A — F + varying pruning (no LoRA)

For each sparsity in {10, 20, 30, 50}%:
- Taylor X% alone → establish the pruning-only baseline at this level
- Taylor X% + F → add F duplication on top

Compare each +F delta to isolate the duplication contribution at each sparsity level.

In [ ]:
phase_a = {}

# Dense baselines (sanity) — weights pristine from load, no restore needed here.
# (The restore_mlp_weights() call was silently perturbing CUDA state/kernel selection
#  even though values were byte-identical. See FINDINGS / diagnostic in section 6.5.)
print("=== Phase A — baselines ===")
reset_layers()
phase_a["dense"], _ = eval_tag("pA_dense")

# Warmup: run a DIFFERENT duplication range before F. This mirrors notebook 6's ordering
# (where 7 other variants ran before F). Tests whether kernel selection / CUDA warmup
# history affects bf16 numerics compounded over 45 layers.
# Range [9, 15] = notebook 6's E variant (43 layers, different from F's 45).
set_duplication_range(9, 15)
phase_a["warmup_E_9_15"], _ = eval_tag("pA_warmup_E_9_15")
reset_layers()

set_F()
phase_a["F_only"], _ = eval_tag("pA_F_only")
reset_layers()

print("\n=== Phase A — prune + (optional F) sweep ===")
for sp in PRUNE_LEVELS:
    restore_mlp_weights()
    apply_taylor_masks(sp)

    # Pruned alone
    reset_layers()
    phase_a[f"prune_{int(sp*100)}"], _ = eval_tag(f"pA_prune_{int(sp*100)}")

    # Pruned + F
    set_F()
    phase_a[f"prune_{int(sp*100)}_F"], _ = eval_tag(f"pA_prune_{int(sp*100)}_F")
    reset_layers()

# Cleanup back to dense state
restore_mlp_weights()
print("\nPhase A done.")

=== Phase A — baselines ===


[pA_dense] MMLU subjects: 100%|██████████████████████████████████████| 10/10 [01:21<00:00,  8.18s/it]


  pA_dense                           MMLU 48.67%   (94.1s)


[pA_warmup_E_9_15] MMLU subjects: 100%|██████████████████████████████| 10/10 [01:37<00:00,  9.72s/it]


  pA_warmup_E_9_15                   MMLU 40.88%   (107.7s)


...

  pA_prune_10                        MMLU 35.65%   (91.0s)


[pA_prune_10_F] MMLU subjects: 100%|█████████████████████████████████| 10/10 [01:40<00:00, 10.02s/it]


  pA_prune_10_F                      MMLU 31.90%   (110.3s)


[pA_prune_20] MMLU subjects: 100%|███████████████████████████████████| 10/10 [01:17<00:00,  7.80s/it]


  pA_prune_20                        MMLU 26.78%   (88.3s)


## 6. Phase A summary

In [ ]:
dense = phase_a["dense"]
print("=" * 80)
print(f"  {'Config':<24s} {'Sparsity':>10s} {'MMLU':>8s} {'Δ vs dense':>12s} {'F-gain':>10s}")
print("-" * 80)
print(f"  {'dense baseline':<24s} {'0%':>10s} {dense*100:>7.2f}% {'-':>12s} {'-':>10s}")
print(f"  {'F only':<24s} {'0%':>10s} {phase_a['F_only']*100:>7.2f}% {(phase_a['F_only']-dense)*100:>+10.2f}pp {'-':>10s}")
for sp in PRUNE_LEVELS:
    k_prune = f"prune_{int(sp*100)}"
    k_prune_f = f"prune_{int(sp*100)}_F"
    d_prune = (phase_a[k_prune] - dense) * 100
    d_prune_f = (phase_a[k_prune_f] - dense) * 100
    f_gain = (phase_a[k_prune_f] - phase_a[k_prune]) * 100
    print(f"  {f'Taylor {int(sp*100)}% alone':<24s} {f'{int(sp*100)}%':>10s} {phase_a[k_prune]*100:>7.2f}% {d_prune:>+10.2f}pp {'-':>10s}")
    print(f"  {f'Taylor {int(sp*100)}% + F':<24s} {f'{int(sp*100)}%':>10s} {phase_a[k_prune_f]*100:>7.2f}% {d_prune_f:>+10.2f}pp {f_gain:>+8.2f}pp")
print("=" * 80)

# Pick the best Taylor X% + F config for Phase B
combo_keys = [f"prune_{int(sp*100)}_F" for sp in PRUNE_LEVELS]
best_key = max(combo_keys, key=lambda k: phase_a[k])
best_sp = int(best_key.split("_")[1]) / 100.0
print(f"\nBest Taylor+F config for Phase B: Taylor {int(best_sp*100)}% + F (MMLU {phase_a[best_key]*100:.2f}%)")

## 6.5 Diagnostic — F_only anomaly

Phase A reported F_only = 38.21% but notebook 6 gets 48.87% on the same config.
Re-run F_only and check identity + weight integrity to localize the issue.

In [ ]:
# 1) Reproduce F_only — is it still 38%?
restore_mlp_weights()
reset_layers()
diag_dense, _ = eval_tag("diag_dense")
set_F()
diag_F, _ = eval_tag("diag_F_only")

# 2) Identity check — are ModuleList positions 8 and 17 the SAME Python object?
print()
print("== identity checks ==")
print(f"len(model.model.layers) = {len(model.model.layers)}  (expect 45)")
print(f"layers[8] is layers[17]  = {model.model.layers[8] is model.model.layers[17]}  (expect True — duplicated same ref)")
print(f"layers[8] is layers[8] (identity sanity) = {model.model.layers[8] is ORIGINAL_LAYERS[8]}  (expect True)")
print(f"layers[17] is ORIGINAL_LAYERS[8] = {model.model.layers[17] is ORIGINAL_LAYERS[8]}  (expect True)")
print(f"layers[26] is ORIGINAL_LAYERS[17] = {model.model.layers[26] is ORIGINAL_LAYERS[17]}  (expect True — post-block)")

# 3) layer_types sync check
print()
print("== config checks ==")
print(f"config.num_hidden_layers = {model.config.num_hidden_layers}  (expect 45)")
if getattr(model.config, 'layer_types', None) is not None:
    print(f"len(config.layer_types)  = {len(model.config.layer_types)}  (expect 45)")
    print(f"config.layer_types[:5]   = {model.config.layer_types[:5]}")

# 4) Weight integrity — compare a sample layer's weight to the cached original
import torch
print()
print("== weight integrity ==")
for probe_name in ["model.layers.0.mlp.gate_proj.weight", "model.layers.8.mlp.gate_proj.weight", "model.layers.17.mlp.gate_proj.weight"]:
    mod = dict(model.named_modules()).get(probe_name[:-len('.weight')])
    if mod is None:
        print(f"  {probe_name}: MODULE NOT FOUND via named_modules (could be deduped out)")
        continue
    cached = ORIGINAL_MLP_WEIGHTS[probe_name]
    live   = mod.weight.data.cpu()
    diff   = (cached - live).abs().max().item()
    zero   = (live == 0).float().mean().item()
    print(f"  {probe_name}: max|cached - live| = {diff:.2e}   zero_frac = {zero*100:.2f}%")

print()
print(f"Reproduced: dense = {diag_dense*100:.2f}%   F_only = {diag_F*100:.2f}%")
print(f"Expected:   dense = 48.67%   F_only = 48.87% (from notebook 6)")
reset_layers()

## 7. Phase B — Best Taylor X% + F + LoRA recovery

Configuration (matches notebook 5's winning setup — the only LoRA config we know is safe on 8 GB):
- r=16, alpha=32, target_modules = {gate_proj, up_proj, down_proj}
- Full-precision `torch.optim.AdamW` (NOT 8-bit — see notebook 5 section 15 for why)
- 500 samples from `cais/mmlu auxiliary_train`, 3 epochs, LR=1e-4
- Answer-token-only loss (IGNORE everywhere except the single A/B/C/D token)
- MAX_LEN 192, `enable_input_require_grads()` for PEFT + gradient checkpointing

Expected peak VRAM: ~7 GB (same as notebook 5 at r=16). The F duplication adds zero params (shared refs) so the only extra memory is activations across 45 layers vs 36 — marginal at batch 1.

In [ ]:
# Build the base: restore weights, apply best Taylor masks, duplicate F
restore_mlp_weights()
apply_taylor_masks(best_sp)
set_F()
print(f"base built: Taylor {int(best_sp*100)}% + F")

# Sanity re-eval the pre-LoRA number (should match phase_a[best_key])
pre_lora_acc, _ = eval_tag(f"pB_pre_lora_{int(best_sp*100)}_F")
print(f"(expected {phase_a[best_key]*100:.2f}%)")

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.0,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
# CRITICAL for grad checkpoint + PEFT on frozen base — see notebook 5 bug fixes
model.enable_input_require_grads()
model.print_trainable_parameters()
print(f"peak GPU after LoRA attach: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

In [ ]:
# Training data — MMLU aux 500 samples, answer-token-only loss (notebook 5 recipe)
N_TRAIN = 500
EPOCHS = 3
LR = 1e-4
MAX_LEN = 192
IGNORE = -100

CHOICE_LABELS = ["A", "B", "C", "D"]
CHOICE_TOKEN_IDS = [tokenizer.encode(lbl, add_special_tokens=False)[-1] for lbl in CHOICE_LABELS]

def format_prompt(question, choices):
    lines = [question]
    for label, choice in zip(CHOICE_LABELS, choices):
        lines.append(f"{label}. {choice}")
    lines.append("Answer:")
    return "\n".join(lines)

mmlu_train = load_dataset("cais/mmlu", "all", split="auxiliary_train",
                          streaming=True, trust_remote_code=True)
train_rows = []
for row in mmlu_train:
    train_rows.append(row)
    if len(train_rows) >= N_TRAIN:
        break

train_batches = []
skipped = 0
for row in train_rows:
    ans_idx = int(row["answer"])
    prompt = format_prompt(row["question"], row["choices"])
    prompt_ids = tokenizer(prompt, return_tensors="pt",
                           max_length=MAX_LEN - 1, truncation=True)["input_ids"][0]
    answer_tok = torch.tensor([CHOICE_TOKEN_IDS[ans_idx]], dtype=prompt_ids.dtype)
    input_ids = torch.cat([prompt_ids, answer_tok])
    if len(input_ids) > MAX_LEN:
        skipped += 1
        continue
    labels = torch.full_like(input_ids, IGNORE)
    labels[-1] = answer_tok.item()
    train_batches.append({
        "input_ids":      input_ids.unsqueeze(0),
        "attention_mask": torch.ones_like(input_ids).unsqueeze(0),
        "labels":         labels.unsqueeze(0),
    })
print(f"prepared {len(train_batches)} batches (skipped {skipped})")
assert len(train_batches) > 0

In [ ]:
model.train()
model.gradient_checkpointing_enable()
model.config.use_cache = False

optim = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
)

gc.collect(); torch.cuda.empty_cache()
print(f"VRAM before training: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

losses = []
t0 = time.time()
for epoch in range(EPOCHS):
    order = np.random.RandomState(42 + epoch).permutation(len(train_batches))
    pbar = tqdm(order, desc=f"epoch {epoch+1}/{EPOCHS}")
    for idx in pbar:
        b = train_batches[idx]
        out = model(
            input_ids=b["input_ids"].to(config.DEVICE),
            attention_mask=b["attention_mask"].to(config.DEVICE),
            labels=b["labels"].to(config.DEVICE),
        )
        out.loss.backward()
        optim.step()
        optim.zero_grad(set_to_none=True)
        losses.append(out.loss.item())
        if len(losses) % 20 == 0:
            pbar.set_postfix(loss=f"{np.mean(losses[-20:]):.3f}")

print(f"\ntrained {len(losses)} steps in {time.time()-t0:.1f}s")
print(f"first 20 loss mean: {np.mean(losses[:20]):.3f}")
print(f"last 20 loss mean:  {np.mean(losses[-20:]):.3f}")
print(f"peak GPU during training: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

In [ ]:
post_lora_acc, _ = eval_tag(f"pB_post_lora_{int(best_sp*100)}_F")
print(f"\nLoRA Δ: {(post_lora_acc - pre_lora_acc)*100:+.2f}pp")

## 8. Final summary — the full RMP picture

In [ ]:
print("=" * 80)
print(f"  FULL RMP — Qwen 2.5 3B, F = layers [{F_RANGE[0]},{F_RANGE[1]}] duplicated")
print("=" * 80)
print(f"  Dense baseline                       {phase_a['dense']*100:6.2f}%")
print(f"  F only (layers 8-16 dup)             {phase_a['F_only']*100:6.2f}%  (Δ {(phase_a['F_only']-dense)*100:+.2f}pp)")
print()
for sp in PRUNE_LEVELS:
    k_prune = f"prune_{int(sp*100)}"
    k_prune_f = f"prune_{int(sp*100)}_F"
    print(f"  Taylor {int(sp*100):2d}% pruning                 {phase_a[k_prune]*100:6.2f}%  (Δ {(phase_a[k_prune]-dense)*100:+.2f}pp)")
    print(f"  Taylor {int(sp*100):2d}% + F duplication         {phase_a[k_prune_f]*100:6.2f}%  (Δ {(phase_a[k_prune_f]-dense)*100:+.2f}pp)")
print()
print(f"  Best Taylor+F config: {int(best_sp*100)}%")
print(f"    pre-LoRA                           {pre_lora_acc*100:6.2f}%")
print(f"    + LoRA (r=16, AdamW fp, MMLU-aux)  {post_lora_acc*100:6.2f}%  (Δ from pre-LoRA: {(post_lora_acc-pre_lora_acc)*100:+.2f}pp)")
print(f"    vs dense                           {(post_lora_acc-dense)*100:+.2f}pp")
print("=" * 80)